# EJEMPLO 2
## Transferencia de Aprendizaje

In [ ]:
import tensorflow as tf
from tensorflow import keras
print("Versión TF", tf.__version__)

Versión TF 2.21.0


**Carga del Dataset**

Carga un dataset de tomates saludables y dañados que ha sido redimensionado de imágenes 800x600 a 224x224. La siguiente es la referencia del dataset original:

> Mishra, S., & Sharma, S. (2026). INDTomatoMSTD: A Multi-Scale Tomato Disease Dataset for Tiny and Large Lesion Detection Using Deep Transfer Learning. INDTomatoMSTD: A Multi-Scale Tomato Disease Dataset for Tiny and Large Lesion Detection Using Deep Transfer Learning. IEEE Dataport. doi:10.21227/2194-m629

Recuerde descargar el dataset (https://drive.google.com/drive/folders/1IJgz_xk4T-HwzW0fUwdVCBOnqWkzPY3q?usp=drive_link) y subirlo a su carpeta local de GoogleDrive.

Puede usar el siguiente prompt para cargar el dataset y partirlo en conjuntos de entrenamiento, validación y prueba:

> Necesito que cargues el dataset que tengo en mi carpeta de googledrive en la ruta "cursos\Taller_Deeplearning_Villavicencio\dataset_resized". El dataset tiene dos clases: Healthy y Rejected. Las imágenes de cada clase están en subcarpetas del mismo nombre. Carga inicialmente los dos dataset, uno por cada clase, crea un vector de etiquetas, y luego realiza una partición de estos dataset de la siguiente forma: entrenamiento 80%, validación 10% y prueba 10% de los datos.

**Crea el modelo basado en MobileNetV3 Small**

Para ello, carga únicamente las primeras capas asociadas a la extracción de características y elimina las capas asociadas a la clasificación. Estas se reemplazan por nuevas capas para hacer la clasificación en dos categorías: Healthy y Rejected.

Puedes usar el siguiente prompt:

> Crea un modelo basado en MobuleNetV3 Small que permita entrenar las capas de clasificación, es decir, cambiar la cabeza del modelo por uno que permita clasificar los dos patrones del dataset. Muestra al final un resumen del número de parámetros del modelo.

**Entrenamiento de las capas de clasificación**

Puedes usar el siguiente prompt:

> Compila y entrena el modelo usando el dataset de entrenamiento y validación usando el algoritmo Adam. Añade las estrategias para evitar overfitting de EarlyStopping y ModelCheckpoint. Grafica las curvas de evolución de accuracy y loss con las épocas para los conjuntos de entrenamiento y validación

**Guarda el modelo**

In [ ]:
model.save("clasificador_tomates_v1.keras")

**Fine-tuning**

Realiza un proceso de reentrenamiento por Fine-Tunning donde las últimas capas del modelo (no solamente las capas de clasificación) son ajustadas. Este proceso mejora la calidad del modelo.

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Entrenar el modelo con los callbacks
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stopping, model_checkpoint]
)

# Graficar el histórico de Accuracy y Loss
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(12, 5))

# Gráfica de Precisión (Accuracy)
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Entrenamiento')
plt.plot(epochs_range, val_acc, label='Validación')
plt.legend(loc='lower right')
plt.title('Precisión de Entrenamiento y Validación')
plt.xlabel('Época')
plt.ylabel('Accuracy')

# Gráfica de Pérdida (Loss)
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Entrenamiento')
plt.plot(epochs_range, val_loss, label='Validación')
plt.legend(loc='upper right')
plt.title('Pérdida de Entrenamiento y Validación')
plt.xlabel('Época')
plt.ylabel('Loss')

plt.tight_layout()
plt.show()

**Exporta el modelo a un formato TFLite FLOAT32**

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

tflite_model = converter.convert()

with open("clasificador_tomates_v1.tflite", "wb") as f:
    f.write(tflite_model)

**Exporta el modelo cuantizado en formato INT8 a TFLite**

In [ ]:
import tensorflow as tf
import numpy as np

# 1. Definir un generador de datos representativos utilizando el dataset existente
def representative_data_gen():
    # Usamos un lote pequeño del dataset para la calibración
    for input_value, _ in val_ds.take(20):
        # Convertimos las imágenes a tipo float32
        yield [tf.cast(input_value, tf.float32)]

# 2. Configurar el convertidor de TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Habilitar optimizaciones
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Configurar el generador de datos representativos obligatorio para INT8
converter.representative_dataset = representative_data_gen

# Restringir las operaciones para asegurar compatibilidad total con TFLite Micro
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

# Configurar entrada y salida del modelo de inferencia en INT8
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

# 3. Convertir el modelo
tflite_quant_model = converter.convert()

# 4. Guardar el archivo cuantizado
tflite_model_path = "clasificador_tomates_quant_int8.tflite"
with open(tflite_model_path, "wb") as f:
    f.write(tflite_quant_model)

print(f"Modelo cuantizado en INT8 exportado exitosamente a: {tflite_model_path}")

In [ ]:
# Script en Python puro para emular el comando xxd y exportar el modelo a C
# Comando original: xxd -i modelo.tflite > modelo.h

def export_tflite_to_c(input_filename, output_filename):
    with open(input_filename, "rb") as f:
        bytes_data = f.read()
    var_name = input_filename.replace(".", "_")
    with open(output_filename, "w") as f:
        f.write(f"// Arreglo de bytes del modelo generado por Python\n")
        f.write(f"const unsigned char {var_name}[] = {{\n    ")
        for i, byte in enumerate(bytes_data):
            f.write(f"0x{byte:02x}")
            if i < len(bytes_data) - 1:
                f.write(", ")
            if (i + 1) % 12 == 0:
                f.write("\n    ")
        f.write("\n};\n\n")
        f.write(f"const unsigned int {var_name}_len = {len(bytes_data)};\n")

    print(f"¡Archivo '{output_filename}' creado con éxito en Python!")

In [ ]:
export_tflite_to_c('clasificador_tomates_v1.tflite', 'clasificador_tomates_v1.h')